In [3]:
import os
from langchain.chains import ConversationalRetrievalChain, LLMChain
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT, QA_PROMPT
from langchain.chains.question_answering import load_qa_chain

from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import time

Provide the API key by running the cell below.

In [4]:
import getpass

if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key

Enter your NVIDIA API key:  ········


Helper functions for loading html files, which we'll use to generate the embeddings. We'll use this later to load the relevant html documents from the Triton documentation website and convert to a vector store.

In [5]:
import re
from typing import List, Union

import requests
from bs4 import BeautifulSoup

def html_document_loader(url: Union[str, bytes]) -> str:
    """
    Loads the HTML content of a document from a given URL and return it's content.

    Args:
        url: The URL of the document.

    Returns:
        The content of the document.

    Raises:
        Exception: If there is an error while making the HTTP request.

    """
    try:
        response = requests.get(url)
        html_content = response.text
    except Exception as e:
        print(f"Failed to load {url} due to exception {e}")
        return ""

    try:
        # Create a Beautiful Soup object to parse html
        soup = BeautifulSoup(html_content, "html.parser")

        # Remove script and style tags
        for script in soup(["script", "style"]):
            script.extract()

        # Get the plain text from the HTML document
        text = soup.get_text()

        # Remove excess whitespace and newlines
        text = re.sub("\s+", " ", text).strip()

        return text
    except Exception as e:
        print(f"Exception {e} while loading document")
        return ""

Read html files and split text in preparation for embedding generation
Note chunk_size value must match the specific LLM used for embedding genetation

Make sure to pay attention to the chunk_size parameter in TextSplitter. Setting the right chunk size is critical for RAG performance, as much of a RAG’s success is based on the retrieval step finding the right context for generation. The entire prompt (retrieved chunks + user query) must fit within the LLM’s context window. Therefore, you should not specify chunk sizes too big, and balance them out with the estimated query size. For example, while OpenAI LLMs have a context window of 8k-32k tokens, Llama3 is limited to 8k tokens. Experiment with different chunk sizes, but typical values should be 100-600, depending on the LLM.

In [7]:
def create_embeddings(embedding_path: str = "./data/nv_embedding"):

    embedding_path = "./data/nv_embedding"
    print(f"Storing embeddings to {embedding_path}")

    # List of web pages containing NVIDIA Triton technical documentation
    urls = [
         "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/index.html",
         "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/getting_started/quickstart.html",
         "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_repository.html",
         "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/model_analyzer.html",
         "https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/user_guide/architecture.html",
    ]

    documents = []
    for url in urls:
        document = html_document_loader(url)
        documents.append(document)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=0,
        length_function=len,
    )
    texts = text_splitter.create_documents(documents)
    index_docs(url, text_splitter, texts, embedding_path)
    print("Generated embedding successfully")

Generate embeddings using NVIDIA AI Endpoints for LangChain and save embeddings to offline vector store in the ./data/nv_embedding directory for future re-use

In [8]:
def index_docs(url: Union[str, bytes], splitter, documents: List[str], dest_embed_dir) -> None:
    """
    Split the document into chunks and create embeddings for the document

    Args:
        url: Source url for the document.
        splitter: Splitter used to split the document
        documents: list of documents whose embeddings needs to be created
        dest_embed_dir: destination directory for embeddings

    Returns:
        None
    """
    embeddings = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5", truncate="END")

    for idx, document in enumerate(documents): 
        texts = splitter.split_text(document.page_content)

        # metadata to attach to document
        metadatas = [document.metadata]

        # create embeddings and add to vector store
        if os.path.exists(dest_embed_dir):
            update = FAISS.load_local(folder_path=dest_embed_dir, embeddings=embeddings, allow_dangerous_deserialization=True)
            update.add_texts(texts, metadatas=metadatas)
            update.save_local(folder_path=dest_embed_dir)
        else:
            docsearch = FAISS.from_texts(texts, embedding=embeddings, metadatas=metadatas)
            docsearch.save_local(folder_path=dest_embed_dir)
        time.sleep(1)
        print(f'Added {idx+1}/{len(documents)} documents', end='\r')

In [9]:
create_embeddings()

Storing embeddings to ./data/nv_embedding
Generated embedding successfully


In [10]:
embedding_model = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5", truncate="END", allow_dangerous_deserialization=True)

Load documents from vector database using FAISS

In [11]:
# Embed documents
embedding_path = "./data/nv_embedding"
docsearch = FAISS.load_local(folder_path=embedding_path, embeddings=embedding_model, allow_dangerous_deserialization=True)
retriever = docsearch.as_retriever()

In [12]:
# This should return documents related to the test query
retriever.invoke("Deploy TensorRT-LLM Engine on Triton Inference Server")

[Document(metadata={}, page_content='NVIDIA Triton Inference Server — NVIDIA Triton Inference Server Skip to main content Back to top Ctrl+K NVIDIA Triton Inference Server GitHub NVIDIA Triton Inference Server GitHub Table of Contents Home Release notes Compatibility matrix Getting Started Quick Deployment Guide by backend TRT-LLM vLLM Python with HuggingFace PyTorch ONNX Openvino LLM With TRT-LLM Multimodal model Stable diffusion Scaling guide Multi-Node (AWS) Multi-Instance LLM Features Constrained Decoding Function Calling Speculative Decoding TRT-LLM vLLM Client API Reference OpenAI API KServe API HTTP/REST and GRPC Protocol Extensions Binary tensor data extension Classification extension Schedule policy extension Sequence extension Shared-memory extension Model configuration extension Model repository extension Statistics extension Trace extension Logging extension Parameters extension In-Process Triton Server API C/C++ Python Kafka I/O Rayserve Java Client Libraries Python triton

In [13]:
print(f"{CONDENSE_QUESTION_PROMPT = }")
print(f"{QA_PROMPT = }")

CONDENSE_QUESTION_PROMPT = PromptTemplate(input_variables=['chat_history', 'question'], input_types={}, partial_variables={}, template='Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.\n\nChat History:\n{chat_history}\nFollow Up Input: {question}\nStandalone question:')
QA_PROMPT = PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}\n\nQuestion: {question}\nHelpful Answer:")


In [14]:
llm = ChatNVIDIA(model='mistralai/mixtral-8x7b-instruct-v0.1')
chat = ChatNVIDIA(model="mistralai/mixtral-8x7b-instruct-v0.1", temperature=0.1, max_tokens=1000, top_p=1.0)

retriever = docsearch.as_retriever()

## Requires question and chat_history
qa_chain = (RunnablePassthrough()
    ## {question, chat_history} -> str
    | CONDENSE_QUESTION_PROMPT | llm | StrOutputParser()
    # | RunnablePassthrough(print)
    ## str -> {question, context}
    | {"question": lambda x: x, "context": retriever}
    # | RunnablePassthrough(print)
    ## {question, context} -> str
    | QA_PROMPT | chat | StrOutputParser()
)

Ask any question about Triton

In [17]:
chat_history = []

query = "What is Triton?"
chat_history += [qa_chain.invoke({"question": query, "chat_history": chat_history})]
chat_history

['Triton, as referred to in the given context, is a server from NVIDIA that allows multiple models and/or multiple instances of the same model to execute in parallel on the same system. It can handle inference requests for these models, managing their execution and batching them for efficiency. It supports custom pre- and post-processing operations or even new deep-learning frameworks. Triton also provides a model management API for querying and controlling the models being served. It has health endpoints and metrics to aid in deployment, such as in Kubernetes. The server can handle sequences of inference requests, scheduling them onto the model instances using the Direct scheduling strategy.']

Ask another question about Triton

In [18]:
query = "What interfaces does Triton support?"
chat_history += [""]
for token in qa_chain.stream({"question": query, "chat_history": chat_history[:-1]}):
    print(token, end="")
    chat_history[-1] += token

Based on the provided context, Triton supports several interfaces for integration. It is available through HTTP/REST or gRPC protocol, as well as a C API and Java API. These interfaces allow Triton to be linked directly into your application for edge and other in-process use cases. Additionally, Triton also provides a model management API for querying and controlling the models being served. This model management API is also available through HTTP/REST or gRPC protocol, or by the C API.

Finally showcase chat capabilites by asking a question about the previous query

In [19]:
query = "But why?"
for token in qa_chain.stream({"question": query, "chat_history": chat_history}):
    print(token, end="")

Based on the provided documents, Triton supports multiple interfaces for integration to provide flexibility and ease of use for different scenarios and applications. It offers a model management API that is available by HTTP/REST or GRPC protocol, or by the C API. This allows users to choose the interface that best fits their needs, whether it's for integration with Kubernetes, direct application linking for edge or in-process use cases, or other requirements. The different interfaces also enable a wider range of developers to use Triton, as they can select the one they are most familiar or comfortable with.

Now we demonstrate a simpler chain using a single LLM only, a chat LLM

In [22]:
chat = ChatNVIDIA(
    model='mistralai/mixtral-8x7b-instruct-v0.1', 
    temperature=0.1, 
    max_tokens=1000, 
    top_p=1.0
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("user", 
        "Use the following pieces of context to answer the question at the end."
        " If you don't know the answer, just say that you don't know, don't try to make up an answer."
        "\n\nHISTORY: {history}\n\n{context}\n\nQuestion: {question}\nHelpful Answer:"
    )
])

## Requires question and chat_history
qa_chain = (
    RunnablePassthrough.assign(context = (lambda state: state.get("question")) | retriever)
    # | RunnablePassthrough(print)
    | qa_prompt | chat | StrOutputParser()
)

Now try asking a question about Triton with the simpler chain. Compare the answer to the result with previous complex chain model

In [23]:
chat_history = []

query = "What is Triton?"
chat_history += [qa_chain.invoke({"question": query, "history": chat_history})]
chat_history

['Based on the provided context, Triton refers to NVIDIA Triton Inference Server. It is an open-source inference serving software that streamlines AI inferencing. Triton enables teams to deploy any AI model from multiple deep learning and machine learning frameworks, including TensorRT, PyTorch, ONNX, OpenVINO, Python, RAPIDS FIL, and more. It delivers optimized performance for various query types and supports inference across different environments like cloud, data center, edge, and embedded devices on NVIDIA GPUs, x86 and ARM CPU, or AWS Inferentia.']

Ask another question about Triton

In [24]:
query = "Does Triton support ONNX?"
chat_history += [""]
for token in qa_chain.stream({"question": query, "history": chat_history[:-1]}):
    print(token, end="")
    chat_history[-1] += token

Yes, Triton does support ONNX. According to the provided context, Triton supports all ONNX models that are supported by the version of ONNX Runtime being used by Triton. However, models will not be supported if they use a stale ONNX opset version or contain operators with unsupported types. An ONNX model can be a single file or a directory containing multiple files, and the default name can be overridden using the default\_model\_filename property in the model configuration.

Finally showcase chat capabilites by asking a question about the previous query

In [25]:
query = "How come?"
for token in qa_chain.stream({"question": query, "history": chat_history}):
    print(token, end="")

I need a specific question to provide a helpful and accurate answer. The provided context contains information about NVIDIA Triton Inference Server, its support for ONNX models, and some details about state tensors, input/output tensors, and instance groups in its configuration. If you have a specific question related to this context, please provide it and I will do my best to answer it.

<img src="./images/DLI_Header.png" width=400/>